In [2]:
from collections import OrderedDict
from scipy.optimize import fmin_l_bfgs_b
from lib.Q3 import (get_loss, 
                    get_gradients, 
                    get_sentences_tags)
import numpy as np
import pandas as pd

In [3]:
emissionParametersdf = pd.read_csv('./data/partial/emissionParameters.csv')
transmissionParametersdf = pd.read_csv('./data/partial/transitionParameters.csv')
transmissionParametersdf = transmissionParametersdf.set_index("feature", inplace=False)
transmissionParamsDict = transmissionParametersdf["weight"].to_dict()
emissionParametersdf = emissionParametersdf.set_index('feature', inplace=False)
emissionParamsDict = emissionParametersdf["weight"].to_dict()


transmissionCountDict=transmissionParametersdf["count[y(i-1),y]"].to_dict()
emissionCountDict = emissionParametersdf["count[y,x]"].to_dict()
tagsdf=pd.concat([transmissionParametersdf['y'],emissionParametersdf['y']])

tags = tagsdf.unique()


# tags
emission_feature_dict=(emissionParametersdf['weight']*emissionParametersdf['count[y,x]']).to_dict()
transition_feature_dict=(transmissionParametersdf['weight']*transmissionParametersdf['count[y(i-1),y]'])

In [4]:
emissionParamsDict.update(transmissionParamsDict)
feature_dict = OrderedDict(emissionParamsDict)

allFeatures = list(feature_dict.keys())
knownFeatures = [feature for feature in allFeatures if not "<unk>" in feature]
FEATURE_VECTOR = knownFeatures
TAGS = np.delete(tags,3)
SENTENCES, SENTENCE_TAGS = get_sentences_tags('./data/partial/train')

In [6]:
'''

FEATURES = list(feature_dict_input.keys())
weights =  list(feature_dict_input.values())
     get_loss(sentences, sentence_tags, tags, feature_dict)
get_gradients(sentences, sentence_tags, feature_dict, tags)

'''
def regularizedLoss(weightVector, 
                    sentences=SENTENCES, 
                    sentence_tags=SENTENCE_TAGS, 
                    tags=TAGS, 
                    featureVector=FEATURE_VECTOR, 
                    eta=0.1):
    if len(weightVector) != len(featureVector):
        raise ValueError("Length of weight Vector doesn't match that of the feature vector")
        
    keys_list = featureVector
    values_list = weightVector
    zip_iterator = zip(keys_list, values_list)
    feature_dict = OrderedDict(zip_iterator)
    
    # print("RL 1")
    loss = get_loss(sentences, sentence_tags, tags, feature_dict)
    # print("RL 2")
    
    if type(weightVector) != np.ndarray:
        w = np.array(weightVector)
    else:
        w = weightVector
        
    regularization = w.dot(w) * eta
    
    return loss + regularization

def grad(weightVector,
        sentences=SENTENCES, 
        sentence_tags=SENTENCE_TAGS, 
        tags=TAGS,
        featureVector=FEATURE_VECTOR):
#     print("grad 1")
    if len(weightVector) != len(featureVector):
        raise ValueError("Length of weight Vector doesn't match that of the feature vector")
        
    keys_list = featureVector
    values_list = weightVector
    zip_iterator = zip(keys_list, values_list)
    feature_dict = OrderedDict(zip_iterator)
    
    # print("grad 1")
    gradients = get_gradients(sentences, sentence_tags, feature_dict, tags)
    # print("grad 2")
#     gradKeys = set(gradients.keys())
#     trainingSetKeys = set(feature_dict.keys())
#     print(f"Num features in training set: {len(trainingSetKeys)}")
#     print(f"Num features in gradKeys set: {len(gradKeys)}")
#     print(f"Grad keys is super set of training keys: {gradKeys.issuperset(trainingSetKeys)}")
#     print(f"training keys is super set of grad keys: {trainingSetKeys.issuperset(gradKeys)}")
#     print(f"Keys in training but not in grad keys: {trainingSetKeys.difference(gradKeys)}")
#     return
    gradVec = OrderedDict()
    for feature in feature_dict.keys():
        gradVec[feature] = gradients.get(feature, -1e9)
    
    gradientVector = np.array(list(gradVec.values()))
    
    return gradientVector

In [7]:
def get_loss_grad(w):
    """
    This function will only be called by "fmin_l_bfgs_b"
    Arg:
        w: weights, numpy array
    Returns:
        loss: loss, float
        grads: gradients, numpy array
    """
    # to be completed by you,
    # based on the modified loss and gradients,
    # with L2 regularization included
    
    loss = regularizedLoss(w)
    grads = grad(w)
    return loss, grads

In [8]:
init_w = np.random.rand(len(FEATURE_VECTOR))
trial = get_loss_grad(init_w)
trial

(23752.020024344412,
 array([-8.87310868e+02, -1.18255168e+02, -1.30281498e+03, ...,
         2.39120003e+01,  9.10256529e-03,  6.30153915e-01]))

In [ ]:
len(FEATURE_VECTOR) # was 13651 including <unk> features

13651

In [9]:
def callbackF(w):
    '''
    This function will only be called by "fmin_l_bfgs_b"
    Arg:
        w: weights, numpy array
    '''
    # print("callback function running")
    loss = regularizedLoss(w)
    print('Loss:{0:.4f}'.format(loss))

init_w = np.zeros(len(FEATURE_VECTOR))
result = fmin_l_bfgs_b(func=regularizedLoss, 
                       x0=init_w, 
                       fprime=grad,
                       pgtol=0.01, 
                       callback=callbackF,
                       maxiter=30)

print(result)

Loss:15736.9719
Loss:9694.8753
Loss:8667.0926
Loss:7812.3029
Loss:7309.1347
Loss:6745.7205
Loss:6023.4069
Loss:5487.2834
Loss:5065.7616
Loss:4888.5838
Loss:4714.0290
Loss:4558.8257
Loss:4303.4114
Loss:4226.7720
Loss:4085.4789
Loss:3988.7444
Loss:3880.4873
Loss:3821.5223
Loss:3759.4729
Loss:3681.5627
Loss:3583.5090
Loss:3541.1361
Loss:3500.4498
Loss:3459.5372
Loss:3429.4040
Loss:3422.2633
Loss:3415.7001
Loss:3409.7010
Loss:3400.6626
Loss:3400.5030
(array([ 4.32405184,  2.77231384,  6.25923061, ..., -0.45183173,
       -0.42040147, -0.34924516]), 3400.503003108228, {'grad': array([ 2.87861858, -1.18863964,  9.56100333, ..., -0.14055492,
        1.93449693,  0.11054234]), 'task': b'STOP: TOTAL NO. of ITERATIONS REACHED LIMIT', 'funcalls': 38, 'nit': 30, 'warnflag': 1})


In [10]:
from joblib import dump

In [12]:
dump(result[0], "after30iter.joblib")

['after30iter.joblib']

```
(array([0., 0., 0., ..., 0., 0., 0.]), 21095.97400273746, 
{
    'grad': array([-7.89552613e+02, -1.81282974e+02, -1.23542694e+03, ...,
        1.18226804e+01,  3.22859907e-02,  6.58333333e-01]), 
    'task': 'ABNORMAL_TERMINATION_IN_LNSRCH', 
    'funcalls': 21, 
    'nit': 0, 
    'warnflag': 2
    }
)
```

In [ ]:
def f(x):
    print("h")
    return x**2, -x
    
def cbF(x):
#     raise Exception
    print(x)
    print("EHHL")
    
fmin_l_bfgs_b(f, 
               10000000, 
               pgtol=0.01, 
               callback=cbF)
# result

h
h
h
h
h
h
h
h
h
h
h
h
h
h


(array([10000000.]),
 array([1.e+14]),
 {'grad': array([-10000000.]),
  'task': 'ABNORMAL_TERMINATION_IN_LNSRCH',
  'funcalls': 14,
  'nit': 0,
  'warnflag': 2})

In [ ]:
def f(x):
    return x**2 , 2*x
g = np.vectorize(f)

def cb(w):
    print(w)

fmin_l_bfgs_b(g, 
            np.array([10,10]),
            pgtol=0.01,
            callback=cb,
            iprint=99)


[6.46446609 6.46446609]
[8.8817842e-16 8.8817842e-16]


(array([8.8817842e-16, 8.8817842e-16]),
 array([7.88860905e-31, 7.88860905e-31]),
 {'grad': array([1.77635684e-15, 1.77635684e-15]),
  'task': 'CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL',
  'funcalls': 4,
  'nit': 2,
  'warnflag': 0})

In [ ]:
# init_w = np.random.rand(len(FEATURE_VECTOR))
# result = get_loss_grad(init_w)
print(result[0])
print(result[1])

[-3.7252903e-09]
[1.38777878e-17]
